In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [2]:
spark = SparkSession.builder.config("spark.driver.memory", "6g").config("spark.driver.maxResultSize", "2g").master("local[*]").appName("traffic_speed_data").getOrCreate()
base_path = "/home/jovyan/work"

In [3]:
df_raw = spark.read.option("header", True).csv(f"{base_path}/data/traffic_speeds/csv/")

df_raw.printSchema()

root
 |-- ID: string (nullable = true)
 |-- SPEED: string (nullable = true)
 |-- TRAVEL_TIME: string (nullable = true)
 |-- STATUS: string (nullable = true)
 |-- DATA_AS_OF: string (nullable = true)
 |-- LINK_ID: string (nullable = true)
 |-- LINK_POINTS: string (nullable = true)
 |-- ENCODED_POLY_LINE: string (nullable = true)
 |-- ENCODED_POLY_LINE_LVLS: string (nullable = true)
 |-- OWNER: string (nullable = true)
 |-- TRANSCOM_ID: string (nullable = true)
 |-- BOROUGH: string (nullable = true)
 |-- LINK_NAME: string (nullable = true)



In [6]:
df_raw.write.parquet(f"{base_path}/data/traffic_speeds/raw_traffic_speeds/")

In [4]:
df_raw = spark.read.parquet(f"{base_path}/data/traffic_speeds/raw_traffic_speeds/")

In [5]:
df_filtered = df_raw \
    .withColumn("DATE", to_timestamp("DATA_AS_OF", "yyyy MMM dd hh:mm:ss a")) \
    .select(col("ID").cast("int"), "DATE", col("SPEED").cast("double"), col("TRAVEL_TIME").cast("int"))

df_filtered.printSchema()

root
 |-- ID: integer (nullable = true)
 |-- DATE: timestamp (nullable = true)
 |-- SPEED: double (nullable = true)
 |-- TRAVEL_TIME: integer (nullable = true)



In [6]:
df_filtered = df_filtered \
    .filter(year(col("DATE")).between(2023, 2025)) \
    .filter((col("SPEED").isNotNull()) & (col("SPEED") > 0.00)) \
    .filter((col("TRAVEL_TIME").isNotNull()) & (col("TRAVEL_TIME") > 0.00))

In [7]:
df_sensors = df_raw \
    .select("ID", "LINK_POINTS", "BOROUGH") \
    .distinct()

In [8]:
df_sensors = df_sensors.dropDuplicates(["ID"])

In [9]:
df_sensors_coords = df_sensors \
    .withColumn(
        "LINK_POINTS_ARRAY",
        split(
            trim(regexp_replace(
                col("LINK_POINTS"),
                r"(\d)(\d{2}\.)",
                r"$1 $2"
            )),
            r"\s+"
        )
    )

In [10]:
df_sensors_coords = df_sensors_coords \
    .withColumn("coord", explode(col("LINK_POINTS_ARRAY"))) \
    .filter(col("coord") != "") \
    .withColumn("lat", split(col("coord"), ",")[0].cast("double")) \
    .withColumn("lon", split(col("coord"), ",")[1].cast("double")) \
    .filter(col("lat").isNotNull() & col("lon").isNotNull()) \
    .withColumn("pos", monotonically_increasing_id())

In [11]:
w = Window.partitionBy("ID").orderBy("pos")

df_sensors_coords = df_sensors_coords \
    .withColumn("prev_lat", lag("lat").over(w)) \
    .withColumn("prev_lon", lag("lon").over(w)) \
    .withColumn("next_lat", lead("lat").over(w)) \
    .withColumn("next_lon", lead("lon").over(w)) \
    .withColumn("dist_prev", sqrt(pow(col("lat") - col("prev_lat"), 2) + pow(col("lon") - col("prev_lon"), 2))) \
    .withColumn("dist_next", sqrt(pow(col("lat") - col("next_lat"), 2) + pow(col("lon") - col("next_lon"), 2)))

In [12]:
THRESHOLD = 4.0
median_dist = df_sensors_coords \
    .groupBy("ID") \
    .agg(percentile_approx("dist_prev", 0.5).alias("median_dist"))  # 0.5 = Median)

df_sensors_coords = df_sensors_coords \
    .join(median_dist, on="ID") \
    .filter(
        (col("dist_prev").isNull() & (col("dist_next") <= col("median_dist") * THRESHOLD)) |
        (col("dist_next").isNull() & (col("dist_prev") <= col("median_dist") * THRESHOLD)) |
        ((col("dist_prev") <= col("median_dist") * THRESHOLD) & (col("dist_next") <= col("median_dist") * THRESHOLD))
    )

In [13]:
df_sensors_coords.printSchema()

root
 |-- ID: string (nullable = true)
 |-- LINK_POINTS: string (nullable = true)
 |-- BOROUGH: string (nullable = true)
 |-- LINK_POINTS_ARRAY: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- coord: string (nullable = false)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- pos: long (nullable = false)
 |-- prev_lat: double (nullable = true)
 |-- prev_lon: double (nullable = true)
 |-- next_lat: double (nullable = true)
 |-- next_lon: double (nullable = true)
 |-- dist_prev: double (nullable = true)
 |-- dist_next: double (nullable = true)
 |-- median_dist: double (nullable = true)



In [14]:
df_sensors = df_sensors_coords \
    .withColumn("coord_clean", concat_ws(",", col("lat"), col("lon"))) \
    .groupBy("ID") \
    .agg(collect_list("coord_clean").alias("LINK_POINTS")) \
    .join(
        df_sensors.drop("LINK_POINTS"),
        on="ID"
    ) \
    .select(df_sensors.columns)

In [15]:
df_filtered = df_filtered \
    .join(df_sensors, on="ID", how="left") 

In [16]:
df_filtered.printSchema()

root
 |-- ID: integer (nullable = true)
 |-- DATE: timestamp (nullable = true)
 |-- SPEED: double (nullable = true)
 |-- TRAVEL_TIME: integer (nullable = true)
 |-- LINK_POINTS: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- BOROUGH: string (nullable = true)



In [17]:
df_filtered.show(5)

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving


Py4JError: An error occurred while calling o239.showString

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving


In [24]:
df_filtered.write.parquet(f"{base_path}/data/traffic_speeds/cleaned_traffic_speeds/")

In [25]:
spark.stop()